# Check 1 -- were the mining settings picked with knowledge of the test data?

Paper mini-experiment. Appendix E.2.3 chose Table E.13 from sweeps over the full
timeline. Here the same settings are re-selected on **W0 only** (CSCAS baseline
train period), split 80/20 in time order into tuning-train / tuning-val.
Logic: `thesis.system_eval.mining_setting_selection`; run with
`shell-scripts/paper/run_mining_setting_selection.sh` (DGX).

Two selection rules are reported:
- **thesis rule**, applied one axis at a time from the Table E.13 anchor (as the
  thesis swept): shallowest benign depth (recall_benign within one bootstrap
  std), shallowest attack depth on the precision_attack plateau (within one
  bootstrap std of the best), min_samples_leaf by precision_attack (ties → larger
  leaf), max_diverse_rounds by downstream F1 (ties → fewer rounds);
- **shared measure**: best tuning-val RF **F1** (fixed 0.5 threshold, the
  walk-forward's own operating point), then the simplest config within one seed
  std. FP/day and PR-AUC are reported alongside. PR-AUC was the first shared
  measure, but the first run (20261110_075621) found it saturated on tuning-val
  (0.992–0.997 across all 54 settings), so it can't separate settings.

Leaf measures are computed **out of sample** on tuning-val (the thesis computed
them in-sample on the mined data).

Note: `max_diverse_rounds = 2` was originally picked by downstream F1 on the
frozen eval subsample (see `baselines/_mining_modes.py`), and
`max_depth_attack = 4` was justified as "matches single_tree's max_depth", not
by the plateau rule -- so a different W0 pick for either is plausible.

In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from thesis.paths import RESULTS_DIR, ROOT
from thesis.system_eval import mining_setting_selection as mss

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 220)


def to_booktabs(df: pd.DataFrame, caption: str, label: str, float_fmt: str = "{:.3f}") -> str:
    styler = df.style.hide(axis="index").format(
        {c: float_fmt for c in df.select_dtypes("float").columns}
    ).format_index(escape="latex", axis=1).format(escape="latex-math", subset=df.select_dtypes("object").columns)
    return styler.to_latex(hrules=True, caption=caption, label=label, position="t", position_float="centering")

## Load

In [ ]:
RUN_DIR = None  # None -> latest full run (results mirror), else a path

def _latest_run() -> Path:
    for base in (RESULTS_DIR / "paper" / "mining-setting-selection",
                 ROOT / "artifacts" / "experiments" / "mining_setting_selection"):
        runs = sorted(d for d in base.glob("*") if (d / "selection_f1.json").exists()) if base.exists() else []
        if runs:
            return runs[-1]
    raise FileNotFoundError("no mining_setting_selection run found")

run_dir = Path(RUN_DIR) if RUN_DIR else _latest_run()
split = json.loads((run_dir / "tuning_split.json").read_text())
stage1 = pd.read_csv(run_dir / "stage1_survivors.csv")
grid = pd.read_csv(run_dir / "stage2_grid.csv")
selection = pd.read_csv(run_dir / "selection.csv")
trace = pd.read_csv(run_dir / "thesis_rule_trace.csv")
print(run_dir)
print(f"{len(grid)} grid points")
split

## Tuning split

If `n_windows_used == 2`, tuning-val had fewer than 50 attacks in W0 alone and
the span was widened to W0 + W1 -- the walk-forward evaluation should then start
from W2.

In [ ]:
print(f"tuning-train: groups [{split['train_start']}, {split['train_end']}) -- {split['n_train_attack']} attacks")
print(f"tuning-val:   groups [{split['train_end']}, {split['val_end']}) -- {split['n_val_attack']} attacks")

## Stage 1 -- growth rate

Survivor counts per min_growth_rate (coverage 0.05). The question: does 3.0 still
sit in a flat region, away from the steep change near 1? `rel_change` = relative
drop in survivors from the previous threshold.

In [ ]:
s1 = stage1.copy()
s1["rel_change"] = s1.groupby("span")["n_survivors"].pct_change()
display(s1)

fig, ax = plt.subplots(figsize=(6, 3.5))
for span, d in s1.groupby("span"):
    ax.plot(d["min_growth_rate"], d["n_survivors"] / d["n_survivors"].max(), marker="o", label=span)
ax.axvline(3.0, color="grey", ls="--", lw=1)
ax.set_xscale("log"); ax.set_xlabel("min_growth_rate"); ax.set_ylabel("survivors (share of max)")
ax.legend(); plt.tight_layout()

## Stage 2 -- per-axis effect (others at the Table E.13 anchor)

In [ ]:
AXES = list(mss.STAGE2_GRID)
METRICS = [("val_recall_benign", "val_recall_benign_boot_std"),
           ("val_precision_attack", "val_precision_attack_boot_std"),
           ("val_f1_mean", "val_f1_std"),
           ("val_fp_per_day_mean", "val_fp_per_day_std"),
           ("val_prauc_mean", "val_prauc_std")]

fig, axes = plt.subplots(len(METRICS), len(AXES), figsize=(14, 12), squeeze=False)
for j, axis in enumerate(AXES):
    sl = grid
    for a in AXES:
        if a != axis:
            sl = sl[sl[a] == mss.THESIS_SETTING[a]]
    sl = sl.sort_values(axis)
    for i, (m, s) in enumerate(METRICS):
        ax = axes[i, j]
        ax.errorbar(sl[axis], sl[m], yerr=sl[s], marker="o", capsize=3)
        ax.axvline(mss.THESIS_SETTING[axis], color="grey", ls="--", lw=1)
        if i == len(METRICS) - 1: ax.set_xlabel(axis)
        if j == 0: ax.set_ylabel(m.replace("val_", "").replace("_mean", ""))
plt.suptitle("tuning-val, others at Table E.13 (dashed = thesis value)"); plt.tight_layout()

### Full grid

In [ ]:
print("spread across settings:", {m: (round(grid[m].min(), 4), round(grid[m].max(), 4)) for m in ("val_f1_mean", "val_fp_per_day_mean", "val_prauc_mean")})
print("median seed std:", {m: round(grid[m].median(), 4) for m in ("val_f1_std", "val_fp_per_day_std", "val_prauc_std")})
grid.sort_values("val_f1_mean", ascending=False)

## Selection

In [ ]:
display(trace)
display(selection)
print(json.loads((run_dir / "selection_f1.json").read_text()))

In [ ]:
tab = selection.rename(columns={
    "setting": "Setting", "thesis_full_timeline": "Full timeline (thesis)",
    "w0_thesis_rule": "W0, thesis rule", "w0_f1": "W0, F1",
}).drop(columns=["same_thesis_rule", "same_f1"])
tab["Setting"] = tab["Setting"].str.replace("_", " ")
print(to_booktabs(tab, "Mining settings re-selected on the first training window only.", "tab:check1-selection", float_fmt="{:g}"))

## Decision

- **Same values** → one sentence in the paper; done.
- **Different values** → step 5: walk-forward with the thesis values vs. the W0
  values, using `w0_settings.yaml` from this run (it holds all three settings):

In [ ]:
differs = not selection["same_thesis_rule"].all() or not selection["same_f1"].all()
print("Any W0 pick differs from Table E.13:", differs)
if differs:
    y = run_dir / "w0_settings.yaml"
    for name in ("check1_thesis_full_timeline", "check1_w0_thesis_rule", "check1_w0_f1"):
        print(f"python src/thesis/scripts/system_eval/run_signature_leakage.py --arms b "
              f"--mining-settings {y} --mining-setting {name}")

## Step 5 -- walk-forward: Table E.13 vs. the W0 pick

The W0 pick differs from Table E.13 (attack-tree depth 2 instead of 4, 1 round,
leaf 20), so both settings run the full walk-forward
(`shell-scripts/paper/run_check1_step5.sh`: baseline_split, γ = 0.05, RF × 5
seeds; arms b = gate off, b_g2 = Check 2's precision gate at g = 2). The newest
run per setting under `check1_step5/` is used.

In [ ]:
from thesis.system_eval import signature_leakage as sl

STEP5_SETTINGS = {"gr3_md1_mda4_rounds2": "Table E.13 (full timeline)",
                  "gr3_md1_mda2_msl20_rounds1": "W0 pick (F1)"}

def _step5_runs() -> dict[str, Path]:
    found = {}
    for base in (RESULTS_DIR / "paper" / "check1-step5", ROOT / "artifacts" / "experiments" / "check1_step5"):
        if not base.exists():
            continue
        for d in sorted(base.glob("*")):
            if not (d / "ablation_summary.csv").exists():
                continue
            cfg = json.loads((d / "config.json").read_text())
            if cfg.get("max_steps") not in (None, "None"):
                continue  # smoke run
            found[cfg["mining_setting"]] = d  # sorted -> newest wins
        if found:
            break
    return found

step5_runs = _step5_runs()
print(step5_runs)
s5 = pd.concat(
    [pd.read_csv(d / "per_step_results.csv").assign(setting=name) for name, d in step5_runs.items()],
    ignore_index=True,
)
s5["key"] = s5["setting"] + " | " + s5["arm"]

In [ ]:
summ = sl.summarize_ablation(s5.assign(arm=s5["key"])).rename(columns={"arm": "key"})
feat = s5.groupby("key")[["n_features", "n_symbolic"]].mean()
summ = summ.merge(feat, left_on="key", right_index=True)
summ[["key", "f1_mean", "f1_std", "pr_auc_mean", "pr_auc_std", "fp_per_day_mean", "fp_per_day_std",
      "recall_unseen_mean", "n_features", "n_symbolic"]]

In [ ]:
TOL = 0.005
rows = []
for arm in sorted(s5["arm"].unique()):
    f1 = s5[s5["arm"] == arm].groupby(["setting", "step_i"])["f1"].mean().unstack(0)
    if set(STEP5_SETTINGS) <= set(f1.columns):
        d = f1["gr3_md1_mda2_msl20_rounds1"] - f1["gr3_md1_mda4_rounds2"]
        rows.append({"arm": arm, "W0 better": int((d > TOL).sum()), "same": int((d.abs() <= TOL).sum()),
                     "W0 worse": int((d < -TOL).sum()), "mean ΔF1 (W0 - E.13)": d.mean()})
pd.DataFrame(rows)

In [ ]:
def pm(df, metric, d=3):
    return df[f"{metric}_mean"].map(f"{{:.{d}f}}".format) + " ± " + df[f"{metric}_std"].map(f"{{:.{d}f}}".format)

lab = summ["key"].str.split(" | ", regex=False, expand=True)
step5_tab = pd.DataFrame({
    "Setting": lab[0].map(STEP5_SETTINGS), "Gate": lab[1].map({"b": "off", "b_g2": "g = 2"}),
    "F1": pm(summ, "f1"), "PR-AUC": pm(summ, "pr_auc"), "FP/day": pm(summ, "fp_per_day", 1),
    "#features": summ["n_features"].map("{:.1f}".format),
})
display(step5_tab)
print(to_booktabs(step5_tab.apply(lambda c: c.str.replace("±", r"$\pm$", regex=False)),
      "Walk-forward with the mining setting chosen on the full timeline (Table E.13) vs. on W0 only "
      "(RF, 5 seeds, $\\gamma=0.05$): mean $\\pm$ std over seeds.", "tab:check1-step5"))

**Decision (doc):** different values → use the W0 values for every paper run if
the walk-forward doesn't contradict them (W0 pick at least as good on F1 /
FP/day), and put the Table E.13 row in the appendix.